<a href="https://colab.research.google.com/github/Raditya-Z/MachineLearning_244107020144/blob/main/JS-06%20-%20ANN/JS_06_Praktikum4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 4

In [4]:
!pip install annoy
!pip install faiss-cpu
!pip install hnswlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 82.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=a13af423aeb6fb95393f6449c4c4859cee7f283c233e591fd9d7fa9eb83b4714
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [5]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")


=== Annoy ===
Build time: 26.86814856529236 detik
Query time: 0.00023937225341796875 detik
Neighbors: [882683, 809312, 589498, 974367, 217289] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.014080524444580078 detik
Query time: 0.012477397918701172 detik
Neighbors: [882683 809312 589498 974367 217289] ...

=== HNSW (hnswlib) ===
Build time: 171.04379558563232 detik
Query time: 0.0009450912475585938 detik
Neighbors: [882683 809312 589498 974367 217289] ...


In [6]:

# Instalasi library
!pip install -q annoy faiss-cpu hnswlib

# Import library
import numpy as np
import time
import pandas as pd
import gc

from annoy import AnnoyIndex
import faiss
import hnswlib

# =========================================
# 1. Membuat dataset 1 juta data, 5 dimensi
# =========================================
np.random.seed(42)

n_data = 1_000_000
dim = 5
k = 10

X = np.random.random((n_data, dim)).astype(np.float32)
query = np.random.random((1, dim)).astype(np.float32)

# Membuat data ternormalisasi untuk metrik Cosine
X_cosine = X.copy()
query_cosine = query.copy()

faiss.normalize_L2(X_cosine)
faiss.normalize_L2(query_cosine)

# Menyimpan hasil percobaan
hasil = []

# =========================================
# 2. Percobaan Euclidean dan Cosine
# =========================================
for metric in ["Euclidean", "Cosine"]:

    print("\n===================================")
    print("METRIK:", metric)
    print("===================================")

    if metric == "Euclidean":
        data = X
        q = query
        annoy_metric = "euclidean"
        hnsw_metric = "l2"
    else:
        data = X_cosine
        q = query_cosine
        annoy_metric = "angular"
        hnsw_metric = "cosine"

    # -------------------------------------
    # A. ANNOY
    # -------------------------------------
    print("\n--- Annoy ---")

    ann_index = AnnoyIndex(dim, annoy_metric)

    start = time.time()

    for i in range(n_data):
        ann_index.add_item(i, data[i].tolist())

    ann_index.build(10)

    ann_build = time.time() - start

    start = time.time()

    ann_neighbors, ann_distances = ann_index.get_nns_by_vector(
        q[0].tolist(), k, include_distances=True
    )

    ann_query = time.time() - start

    print("Build time:", round(ann_build, 4), "detik")
    print("Query time:", round(ann_query * 1000, 4), "ms")
    print("Neighbors:", ann_neighbors[:5])

    hasil.append({
        "Metrik": metric,
        "Metode": "Annoy",
        "Build (detik)": round(ann_build, 4),
        "Query (ms)": round(ann_query * 1000, 4),
        "5 tetangga terdekat": ann_neighbors[:5]
    })

    del ann_index
    gc.collect()

    # -------------------------------------
    # B. FAISS INDEXFLAT
    # -------------------------------------
    print("\n--- FAISS IndexFlat ---")

    if metric == "Euclidean":
        faiss_index = faiss.IndexFlatL2(dim)
    else:
        faiss_index = faiss.IndexFlatIP(dim)

    start = time.time()
    faiss_index.add(data)
    faiss_build = time.time() - start

    start = time.time()
    distances, indices = faiss_index.search(q, k)
    faiss_query = time.time() - start

    print("Build time:", round(faiss_build, 4), "detik")
    print("Query time:", round(faiss_query * 1000, 4), "ms")
    print("Neighbors:", indices[0][:5])

    hasil.append({
        "Metrik": metric,
        "Metode": "FAISS IndexFlat",
        "Build (detik)": round(faiss_build, 4),
        "Query (ms)": round(faiss_query * 1000, 4),
        "5 tetangga terdekat": indices[0][:5].tolist()
    })

    del faiss_index
    gc.collect()

    # -------------------------------------
    # C. HNSW
    # -------------------------------------
    print("\n--- HNSW ---")

    hnsw_index = hnswlib.Index(
        space=hnsw_metric,
        dim=dim
    )

    start = time.time()

    hnsw_index.init_index(
        max_elements=n_data,
        ef_construction=200,
        M=16
    )

    hnsw_index.add_items(data)
    hnsw_index.set_ef(50)

    hnsw_build = time.time() - start

    start = time.time()
    labels, distances = hnsw_index.knn_query(q, k=k)
    hnsw_query = time.time() - start

    print("Build time:", round(hnsw_build, 4), "detik")
    print("Query time:", round(hnsw_query * 1000, 4), "ms")
    print("Neighbors:", labels[0][:5])

    hasil.append({
        "Metrik": metric,
        "Metode": "HNSW",
        "Build (detik)": round(hnsw_build, 4),
        "Query (ms)": round(hnsw_query * 1000, 4),
        "5 tetangga terdekat": labels[0][:5].tolist()
    })

    del hnsw_index
    gc.collect()

# =========================================
# 3. Membuat tabel hasil praktikum
# =========================================
tabel_hasil = pd.DataFrame(hasil)

print("\nTABEL HASIL PRAKTIKUM 4")
display(tabel_hasil)



METRIK: Euclidean

--- Annoy ---
Build time: 21.6731 detik
Query time: 0.1695 ms
Neighbors: [901095, 495168, 561017, 163986, 185734]

--- FAISS IndexFlat ---
Build time: 0.0296 detik
Query time: 27.6725 ms
Neighbors: [901095 495168 561017 163986 185734]

--- HNSW ---
Build time: 178.6321 detik
Query time: 0.1481 ms
Neighbors: [901095 495168 561017 163986 185734]

METRIK: Cosine

--- Annoy ---
Build time: 28.0065 detik
Query time: 0.2513 ms
Neighbors: [653774, 135588, 561017, 901095, 211473]

--- FAISS IndexFlat ---
Build time: 0.006 detik
Query time: 8.8131 ms
Neighbors: [653774 135588 561017 901095 211473]

--- HNSW ---
Build time: 172.6924 detik
Query time: 0.2027 ms
Neighbors: [653774 135588 561017 901095 211473]

TABEL HASIL PRAKTIKUM 4


,Metrik,Metode,Build (detik),Query (ms),5 tetangga terdekat
0,Euclidean,Annoy,21.6731,0.1695,"[901095, 495168, 561017, 163986, 185734]"
1,Euclidean,FAISS IndexFlat,0.0296,27.6725,"[901095, 495168, 561017, 163986, 185734]"
2,Euclidean,HNSW,178.6321,0.1481,"[901095, 495168, 561017, 163986, 185734]"
3,Cosine,Annoy,28.0065,0.2513,"[653774, 135588, 561017, 901095, 211473]"
4,Cosine,FAISS IndexFlat,0.0060,8.8131,"[653774, 135588, 561017, 901095, 211473]"
5,Cosine,HNSW,172.6924,0.2027,"[653774, 135588, 561017, 901095, 211473]"


# Analisis
> Berdasarkan hasil percobaan, FAISS IndexFlat memiliki waktu pembangunan indeks paling singkat, tetapi waktu pencariannya paling lama. Hal ini karena FAISS IndexFlat mencari tetangga terdekat secara eksak dengan membandingkan query terhadap seluruh data.

> Annoy dan HNSW memiliki waktu pencarian yang jauh lebih singkat, tetapi membutuhkan waktu lebih lama untuk membangun indeks. HNSW memiliki waktu pembangunan indeks paling lama, tetapi memberikan waktu pencarian tercepat pada pengujian Euclidean maupun Cosine.

> Pada percobaan ini, ketiga metode menghasilkan lima indeks tetangga terdekat yang sama untuk masing-masing metrik. Namun, hasil yang sama pada satu percobaan tidak menjamin bahwa Annoy dan HNSW akan selalu memberikan hasil identik dengan metode eksak.

# Kesimpulan
> Dari praktikum ini, dapat disimpulkan bahwa FAISS IndexFlat cocok untuk pencarian eksak dengan proses pembangunan indeks yang cepat, sedangkan Annoy dan HNSW lebih unggul dalam kecepatan pencarian. HNSW memberikan waktu pencarian tercepat pada percobaan ini, tetapi membutuhkan waktu pembangunan indeks paling lama. Oleh karena itu, pemilihan metode bergantung pada kebutuhan, terutama apakah lebih mengutamakan kecepatan pencarian atau waktu persiapan indeks.